In [1]:
import sys
sys.path.append('..')
import correlate

In [2]:
flow = correlate.read_flows('flows_2.mitm')
flow

[{'index': 0,
  'type': 'HTTPFlow',
  'ts': 1794852052.5221426,
  'client_peer': ['192.168.68.107', 14580],
  'client_sock': ['52.21.130.202', 443],
  'sni': 'api2.cursor.sh',
  'host': 'api2.cursor.sh',
  'method': 'POST',
  'path': '/aiserver.v1.AgentStoreService/CheckAgentStoreVersions',
  'scheme': 'https',
  'status': 200,
  'body': '\n\x94\x05\n\x010\x12*store-d6febd33-9d44-4d2c-94a1-615f01bf0168"Í\x04eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJhdWQiOiJhZ2VudC1zdG9yZSIsInN0b3JlSWRzIjpbInN0b3JlLWQ2ZmViZDMzLTlkNDQtNGQyYy05NGExLTYxNWYwMWJmMDE2OCJdLCJjYXBhYmlsaXRpZXMiOlsicmVhZCIsIndyaXRlIl0sInVzZXJJZCI6MzA1OTkxOTE5LCJzZXJ2aWNlQWNjb3VudElkIjpudWxsLCJ0ZWFtSWQiOm51bGwsImIiOnsicyI6InN0b3JlLWQ2ZmViZDMzLTlkNDQtNGQyYy05NGExLTYxNWYwMWJmMDE2OCIsImsiOiJVU0VSIiwiaSI6InUzMDU5OTE5MTkiLCJvIjp7InUiOjMwNTk5MTkxOSwiYSI6bnVsbCwidCI6bnVsbH0sImgiOiJnb29nbGUtb2F1dGgyfHVzZXJfMDFLQVgzUkVEQlQyTlNKVEcwQ0hDUFIySjIiLCJ6IjoxfSwiaWF0IjoxNzk0ODQ5MDEyLCJleHAiOjE3OTQ4Nzc4MTJ9.2hiTR7uNxa_vpRvInoU3tGncvlUufuGXirYXIXujAME

In [3]:
etl = correlate.read_etl('network-stacks.etl')
matched, unmatched = correlate.correlate(flow, etl)

Converting ETL with tracerpt...
ETL window 2026-11-16T18:00:39.862738+00:00 to 2026-11-16T18:01:30.338592+00:00
  100000 events, 244 connections
  200000 events, 573 connections
Parsed 242682 events, 685 process records, 573 TCP rows


In [4]:
import pandas as pd

df = pd.DataFrame(
    {
        "category": correlate._section(row),
        "applicationName": correlate._label(row),
        "host": row["host"],
        "library": row["caller"],
        "target": row["destination_kind"],
        "method": row["method"],
        "path": row["path"],
        "query": row["query"],
        "post body": row["body"],
        "referrer": row["referer"],
    }
    for row in matched
)
tables = {
    category: group.drop(columns="category").reset_index(drop=True)
    for category, group in df.groupby("category")
}
list(tables)

['First-party: own service',
 'Own binary calling another vendor',
 'Third-party: WebView',
 "Third-party: another vendor's module"]

In [5]:
for category, table in tables.items():
    print(category)
    display(table)

First-party: own service


,applicationName,host,library,target,method,path,query,post body,referrer
0,Cursor.exe,api2.cursor.sh,cursor.exe,own service,POST,/aiserver.v1.AgentStoreService/CheckAgentStore...,,\n\n0*store-d6febd33-9d44-4d2c-94a1-615f01...,
1,backgroundTaskHost.exe,www.microsoft.com,backgroundTaskHost.exe,own service,GET,/mitmproxy-43562620863237610265030499034782871...,,,
2,backgroundTaskHost.exe,server.events.data.microsoft.com,backgroundTaskHost.exe,own service,POST,/OneCollector/1.0/,,")3.0Ihealth_telemetry_otherq®©µÝñ¢ß©""o:54f...",
3,backgroundTaskHost.exe,server.events.data.microsoft.com,backgroundTaskHost.exe,own service,POST,/OneCollector/1.0/,,)3.0I health_telemetry_authenticationq´áßñ¢...,
4,backgroundTaskHost.exe,server.events.data.microsoft.com,backgroundTaskHost.exe,own service,POST,/OneCollector/1.0/,,)3.0I&health_telemetry_sync_communication_v2q...,
5,Cursor.exe,api2.cursor.sh,cursor.exe,own service,POST,/aiserver.v1.DashboardService/GetTeams,,,
6,Cursor.exe,api2.cursor.sh,cursor.exe,own service,GET,/auth/full_stripe_profile,,,
7,bdservicehost.exe,push.bitdefender.net,bdnc.dll,own service,GET,/poll,push_id=18fd84c4-b3fc-4ba9-bea7-380ec0deaa78,,
8,bdservicehost.exe,eu.nimbus.bitdefender.net,bdnc.dll,own service,POST,/qid/heur,,QHZÏ¿ôs\räzºã[Ëx/TA w´ª´ ¸ÚÀR$´ú3´Æ@oÎ...,
9,bdservicehost.exe,eu.nimbus.bitdefender.net,bdnc.dll,own service,POST,/qid/fastscan,,ZÏ¿ôs\räzºã[Ëx/TA w´ª´ ¸ÚÀi   %PROGRAM...,


Own binary calling another vendor


,applicationName,host,library,target,method,path,query,post body,referrer
0,WebcamRecorderClient.exe,analytics.google.com,WebcamRecorderClient.exe,other service,POST,/g/collect,v=2&tid=G-5LTJS0NGBQ&cid=88cf054e-15bf-4353-a0...,,
1,WebcamRecorderClient.exe,analytics.google.com,WebcamRecorderClient.exe,other service,POST,/g/collect,v=2&tid=G-5LTJS0NGBQ&cid=88cf054e-15bf-4353-a0...,,
2,WebcamRecorderClient.exe,analytics.google.com,WebcamRecorderClient.exe,other service,POST,/g/collect,v=2&tid=G-5LTJS0NGBQ&cid=88cf054e-15bf-4353-a0...,,
3,WebcamRecorderClient.exe,analytics.google.com,WebcamRecorderClient.exe,other service,POST,/g/collect,v=2&tid=G-5LTJS0NGBQ&cid=88cf054e-15bf-4353-a0...,,
4,WebcamRecorderClient.exe,analytics.google.com,WebcamRecorderClient.exe,other service,POST,/g/collect,v=2&tid=G-5LTJS0NGBQ&cid=88cf054e-15bf-4353-a0...,,
5,WebcamRecorderClient.exe,www.microsoft.com,WebcamRecorderClient.exe,other service,GET,/mitmproxy-43562620863237610265030499034782871...,,,
6,WebcamRecorderClient.exe,displaycatalog.mp.microsoft.com,WebcamRecorderClient.exe,other service,GET,/v7/products/lookup,value=9NKRKS354G33&market=US&languages=en-US&$...,,
7,WebcamRecorderClient.exe,analytics.google.com,WebcamRecorderClient.exe,other service,POST,/g/collect,v=2&tid=G-5LTJS0NGBQ&cid=88cf054e-15bf-4353-a0...,,
8,WebcamRecorderClient.exe,analytics.google.com,WebcamRecorderClient.exe,other service,POST,/g/collect,v=2&tid=G-5LTJS0NGBQ&cid=88cf054e-15bf-4353-a0...,,
9,WebcamRecorderClient.exe,analytics.google.com,WebcamRecorderClient.exe,other service,POST,/g/collect,v=2&tid=G-5LTJS0NGBQ&cid=88cf054e-15bf-4353-a0...,,


Third-party: WebView


,applicationName,host,library,target,method,path,query,post body,referrer
0,SearchHost.exe via msedgewebview2.exe,www.bing.com,msedge.dll,own service,GET,/manifest/threshold.appcache,,,https://www.bing.com/AS/API/WindowsCortanaPane...
1,SearchHost.exe via msedgewebview2.exe,www.bing.com,msedge.dll,own service,POST,/web/xlsc.aspx,t=5&dl=1&f=9&wsbc=1,nVJdT4MwFP0r2OdJKVC+0pAgMINOs60sM1n2UKFzRAazdK...,https://www.bing.com/WS/Init
2,SearchHost.exe via msedgewebview2.exe,storeedge.microsoft.com,msedge.dll,own service,GET,/v9.0/store/products/9PGM3QB3PDRD,market=US&locale=en-US,,https://www.bing.com/
3,SearchHost.exe via msedgewebview2.exe,storeedge.microsoft.com,msedge.dll,own service,GET,/v9.0/store/products/9WZDNCRFJBBG,market=US&locale=en-US,,https://www.bing.com/
4,SearchHost.exe via msedgewebview2.exe,storeedge.microsoft.com,msedge.dll,own service,GET,/v9.0/store/products/9N6J4RNFPKKL,market=US&locale=en-US,,https://www.bing.com/
5,SearchHost.exe via msedgewebview2.exe,storeedge.microsoft.com,msedge.dll,own service,GET,/v9.0/store/products/9WZDNCRFJBBG,market=US&locale=en-US,,https://www.bing.com/
6,SearchHost.exe via msedgewebview2.exe,storeedge.microsoft.com,msedge.dll,own service,GET,/v9.0/store/products/9WZDNCRFJBBG,market=US&locale=en-US,,https://www.bing.com/
7,SearchHost.exe via msedgewebview2.exe,storeedge.microsoft.com,msedge.dll,own service,GET,/v9.0/store/products/9WZDNCRFJBBG,market=US&locale=en-US,,https://www.bing.com/
8,SearchHost.exe via msedgewebview2.exe,dns.google,msedge.dll,own service,GET,/dns-query,dns=AAABAAABAAAAAAABCXN0b3JlZWRnZQltaWNyb3NvZn...,,
9,SearchHost.exe via msedgewebview2.exe,dns.google,msedge.dll,own service,GET,/dns-query,dns=AAABAAABAAAAAAABCXN0b3JlZWRnZQltaWNyb3NvZn...,,


Third-party: another vendor's module


,applicationName,host,library,target,method,path,query,post body,referrer
0,svchost.exe -s LicenseManager,licensing.mp.microsoft.com,svchost.exe,own service,POST,/v7.0/licenses/leases/renew,,"{""clientChallenge"":""PD94bWwgdmVyc2lvbj0iMS4wIi...",
1,ProductAgentService.exe,push.bitdefender.net,wow64cpu.dll,own service,GET,/poll,push_id=44a434c3-c82e-4a32-bec7-e82ae819d731,,
2,msedge.exe,dns.google,msedge.dll,other service,GET,/dns-query,dns=AAABAAABAAAAAAABBGVkZ2UJbWljcm9zb2Z0A2NvbQ...,,
3,msedge.exe,dns.google,msedge.dll,other service,GET,/dns-query,dns=AAABAAABAAAAAAABBGVkZ2UJbWljcm9zb2Z0A2NvbQ...,,
4,msedge.exe,edge.microsoft.com,msedge.dll,own service,GET,/extensionrevocation/v1/threatListUpdates:fetch,req=ChcKBm1zZWRnZRINMTU3LjAuNDM0MS42MBobCAEQCB...,,
